# Hallazgos del EDA y candidatas de feature engineering

Este notebook no cierra todavía la ingeniería de características. Su objetivo es sintetizar los hallazgos del EDA principal, fijar la base de trabajo para el modelado posterior y definir transformaciones candidatas que deberán evaluarse dentro de pipelines reproducibles.

La idea es separar con claridad tres niveles: lo observado en el EDA, lo que se decide depurar antes del split y lo que queda como candidata a transformación posterior.

## 1. Objetivo del notebook

A partir del análisis inicial, se revisan los principales hallazgos del conjunto de datos y se proponen transformaciones que pueden aportar capacidad predictiva o estabilidad al modelado.

Este notebook sirve como puente entre el EDA y los experimentos de clasificación.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.max_colwidth', 120)

BASE_DIR = Path('..')
DATA_PATH = BASE_DIR / 'data' / 'raw' / 'BBDD_ML_TAREA.csv'
TABLES_PATH = BASE_DIR / 'outputs' / 'tables'
RANDOM_STATE = 42

DATA_PATH.resolve()

WindowsPath('D:/Copias de seguridad/2025-2026/Master UCM/Módulo 11/Tarea/data/raw/BBDD_ML_TAREA.csv')

## 2. Carga de datos y tablas del EDA principal

Se carga el dataset original y se recuperan las tablas exportadas desde el notebook de EDA para mantener trazabilidad entre análisis y decisiones.

In [2]:
df_raw = pd.read_csv(DATA_PATH, na_values=['NA'])
quality_summary = pd.read_csv(TABLES_PATH / 'fase1_quality_summary.csv', index_col=0)
target_distribution = pd.read_csv(TABLES_PATH / 'fase1_target_distribution.csv', index_col=0)
redundancy_summary = pd.read_csv(TABLES_PATH / 'fase1_redundancy_summary.csv')
methodological_decisions = pd.read_csv(TABLES_PATH / 'fase1_methodological_decisions.csv')
redundancy_decisions = pd.read_csv(TABLES_PATH / 'fase1_redundancy_decisions.csv')
id_check = pd.read_csv(TABLES_PATH / 'fase1_id_check.csv', index_col=0)

print('Forma del dataset bruto:', df_raw.shape)
display(quality_summary.head())
display(target_distribution)
display(redundancy_summary)
display(methodological_decisions)
display(redundancy_decisions.head())

Forma del dataset bruto: (9200, 21)


,missing_count,missing_pct,n_unique,dtype
V1,0,0.00,51,int64
V10,26,0.28,1742,float64
V11,0,0.00,1646,float64
V12,0,0.00,118,int64
V13,0,0.00,1460,float64


,count,percentage
Y,,
0,4600,50.0
1,4600,50.0


,pair,pearson_corr
0,V8-V10,1.000000
1,V11-V13,1.000000
2,V14-V16,0.999999
3,V17-V19,0.999993


,decision,value,detail
0,deduplicate_before_split,True,5662 duplicados eliminados
1,exclude_V4_by_design,True,V4 excluida
2,redundancy_rule,abs(corr) > 0.999,"Variables eliminadas: ['V10', 'V13', 'V16', 'V19']"
3,dataset_shape_transition,"(9200, 21) -> (3538, 21) -> (3538, 16)",Original -> deduplicado -> modelado


,pair,corr,threshold,drop_cost_variable,kept,dropped_if_applied
0,V8-V10,1.000000,0.999,True,V8,V10
1,V11-V13,1.000000,0.999,True,V11,V13
2,V14-V16,0.999999,0.999,True,V14,V16
3,V17-V19,0.999993,0.999,True,V17,V19


## 3. Hallazgos principales del EDA

Los resultados más relevantes del EDA principal son los siguientes:

- El problema está perfectamente balanceado en la variable objetivo.
- Los valores perdidos existen, pero aparecen en baja proporción.
- Se detectan duplicados exactos en gran número, por lo que la deduplicación previa al split es metodológicamente necesaria.
- Las variables de minutos y coste presentan redundancia casi determinista.
- `V4` se comporta como identificador y no como predictor con sentido sustantivo.

In [3]:
summary_stats = pd.DataFrame({
    'missing_count': quality_summary['missing_count'],
    'missing_pct': quality_summary['missing_pct'],
    'n_unique': quality_summary['n_unique']
})

display(summary_stats.loc[['V3', 'V10', 'V14', 'V15', 'V19']])
display(target_distribution)
display(id_check.head(10))
display(redundancy_summary)

,missing_count,missing_pct,n_unique
V3,75,0.82,3
V10,26,0.28,1742
V14,72,0.78,1622
V15,60,0.65,124
V19,44,0.48,166


,count,percentage
Y,,
0,4600,50.0
1,4600,50.0


,n_unique,unique_ratio
V4,3536,0.3843
V8,1744,0.1896
V10,1743,0.1895
V11,1646,0.1789
V14,1623,0.1764
V13,1460,0.1587
V16,951,0.1034
V2,214,0.0233
V19,167,0.0182
V17,166,0.0180


,pair,pearson_corr
0,V8-V10,1.000000
1,V11-V13,1.000000
2,V14-V16,0.999999
3,V17-V19,0.999993


## 4. Conjunto de modelado base

Antes de definir nuevas variables, se construye el conjunto de trabajo que servirá de base para las fases posteriores: deduplicación, exclusión de `V4` y eliminación de variables de coste redundantes.

Este bloque no introduce todavía feature engineering; solo deja una base limpia y coherente para evaluar transformaciones posteriores.

In [4]:
df_work = df_raw.drop_duplicates().copy()
cols_to_drop = ['V4', 'V10', 'V13', 'V16', 'V19']
df_model_base = df_work.drop(columns=cols_to_drop).copy()

print('Forma del dataset deduplicado:', df_work.shape)
print('Forma de la base de modelado:', df_model_base.shape)
print('Columnas de la base de modelado:')
print(list(df_model_base.columns))

Forma del dataset deduplicado: (3538, 21)
Forma de la base de modelado: (3538, 16)
Columnas de la base de modelado:
['V1', 'V2', 'V3', 'V5', 'V6', 'V7', 'V8', 'V9', 'V11', 'V12', 'V14', 'V15', 'V17', 'V18', 'V20', 'Y']


## 5. Feature engineering candidata

A partir de los hallazgos del EDA, se proponen transformaciones que podrían aportar valor en los modelos posteriores. No se consideran todavía definitivas; deben validarse dentro de pipelines y sin usar el conjunto de prueba para decidir su adopción.

In [5]:
feature_candidates = pd.DataFrame([
    {
        'feature': 'total_minutes',
        'definition': 'V8 + V11 + V14 + V17',
        'motivation': 'Resume la intensidad global de uso sin depender de la franja horaria.'
    },
    {
        'feature': 'total_calls',
        'definition': 'V9 + V12 + V15 + V18',
        'motivation': 'Agrega el volumen total de contactos por franjas.'
    },
    {
        'feature': 'minutes_per_call',
        'definition': 'total_minutes / total_calls',
        'motivation': 'Captura la intensidad media de cada contacto.'
    },
    {
        'feature': 'international_share_minutes',
        'definition': 'V17 / total_minutes',
        'motivation': 'Mide la proporción de uso internacional dentro del total.'
    },
    {
        'feature': 'log1p_v2',
        'definition': 'log(1 + V2)',
        'motivation': 'Reduce asimetría y estabiliza el efecto de la antigüedad.'
    },
    {
        'feature': 'log1p_total_minutes',
        'definition': 'log(1 + total_minutes)',
        'motivation': 'Suaviza colas largas en una variable de uso agregada.'
    },
    {
        'feature': 'customer_service_intensity',
        'definition': 'V20',
        'motivation': 'Mantiene la señal de insatisfacción observable en el número de llamadas al servicio.'
    }
])
display(feature_candidates)

feature_candidates.to_csv(TABLES_PATH / 'fase2_feature_candidates.csv', index=False)
print('Tabla de candidatas exportada en:', TABLES_PATH.resolve())

,feature,definition,motivation
0,total_minutes,V8 + V11 + V14 + V17,Resume la intensidad global de uso sin depender de la franja horaria.
1,total_calls,V9 + V12 + V15 + V18,Agrega el volumen total de contactos por franjas.
2,minutes_per_call,total_minutes / total_calls,Captura la intensidad media de cada contacto.
3,international_share_minutes,V17 / total_minutes,Mide la proporción de uso internacional dentro del total.
4,log1p_v2,log(1 + V2),Reduce asimetría y estabiliza el efecto de la antigüedad.
5,log1p_total_minutes,log(1 + total_minutes),Suaviza colas largas en una variable de uso agregada.
6,customer_service_intensity,V20,Mantiene la señal de insatisfacción observable en el número de llamadas al servicio.


Tabla de candidatas exportada en: D:\Copias de seguridad\2025-2026\Master UCM\Módulo 11\Tarea\outputs\tables


## 6. Próximos pasos

Las transformaciones listadas aquí no deben aplicarse de forma ciega. El siguiente paso consiste en definir, para cada familia de modelos, qué preprocesamiento y qué variables derivadas se justifican empíricamente.

En particular, regresión logística y red neuronal necesitarán codificación e imputación dentro de pipeline, mientras que el árbol permitirá una lectura más directa de la importancia relativa de las variables base.

## 7. Plantilla de evaluacion de candidatas (operativa)

Esta seccion replica en codigo las reglas del protocolo experimental para aceptar o rechazar variables candidatas de feature engineering.

La decision se toma comparando modelo base vs modelo extendido en validacion cruzada sobre train_val.

In [9]:
# Parametros de decision (alineados con 04_protocolo_experimental.tex)
RECALL_TOLERANCE_ACC = 0.02
PRECISION_TOLERANCE_REC = 0.02
PRAUC_TOLERANCE_REC = 0.02
STD_INCREASE_LIMIT = 0.20  # 20%

# Plantilla de resultados por candidata y por objetivo
required_cols = [
    'candidate',
    'objective',  # accuracy o recall
    'acc_base_mean', 'acc_ext_mean',
    'recall_base_mean', 'recall_ext_mean',
    'precision_base_mean', 'precision_ext_mean',
    'prauc_base_mean', 'prauc_ext_mean',
    'primary_std_base', 'primary_std_ext'
]

objectives = ['accuracy', 'recall']
feature_eval_template = pd.MultiIndex.from_product(
    [feature_candidates['feature'], objectives],
    names=['candidate', 'objective']
).to_frame(index=False)

for col in required_cols:
    if col not in feature_eval_template.columns:
        feature_eval_template[col] = np.nan

# Orden de columnas consistente
feature_eval_template = feature_eval_template[required_cols]

display(feature_eval_template)

feature_eval_template.to_csv(TABLES_PATH / 'fase2_feature_eval_template.csv', index=False)
print('Plantilla exportada en:', (TABLES_PATH / 'fase2_feature_eval_template.csv').resolve())
print('Nota: rellena las metricas *_base_* y *_ext_* antes de ejecutar la celda de decision.')

,candidate,objective,acc_base_mean,acc_ext_mean,recall_base_mean,recall_ext_mean,precision_base_mean,precision_ext_mean,prauc_base_mean,prauc_ext_mean,primary_std_base,primary_std_ext
0,total_minutes,accuracy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,total_minutes,recall,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,total_calls,accuracy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,total_calls,recall,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,minutes_per_call,accuracy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,minutes_per_call,recall,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,international_share_minutes,accuracy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,international_share_minutes,recall,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,log1p_v2,accuracy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,log1p_v2,recall,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Plantilla exportada en: D:\Copias de seguridad\2025-2026\Master UCM\Módulo 11\Tarea\outputs\tables\fase2_feature_eval_template.csv
Nota: rellena las metricas *_base_* y *_ext_* antes de ejecutar la celda de decision.


In [7]:
def decide_feature_candidate(row):
    """Aplica las reglas del protocolo y devuelve decision + justificacion."""
    required = [
        'objective',
        'acc_base_mean', 'acc_ext_mean',
        'recall_base_mean', 'recall_ext_mean',
        'precision_base_mean', 'precision_ext_mean',
        'prauc_base_mean', 'prauc_ext_mean',
        'primary_std_base', 'primary_std_ext'
    ]

    # Si faltan metricas, la candidata queda pendiente de evaluacion
    if pd.isna(row[required]).any():
        return pd.Series({'decision': 'pendiente', 'reason': 'faltan metricas de validacion'})

    std_ratio = (row['primary_std_ext'] / row['primary_std_base']) if row['primary_std_base'] > 0 else np.inf
    unstable = std_ratio > (1 + STD_INCREASE_LIMIT)

    if row['objective'] == 'accuracy':
        primary_ok = row['acc_ext_mean'] > row['acc_base_mean']
        guardrail_ok = (row['recall_base_mean'] - row['recall_ext_mean']) <= RECALL_TOLERANCE_ACC
    elif row['objective'] == 'recall':
        primary_ok = row['recall_ext_mean'] > row['recall_base_mean']
        guardrail_ok = (
            (row['precision_base_mean'] - row['precision_ext_mean']) <= PRECISION_TOLERANCE_REC
            and (row['prauc_base_mean'] - row['prauc_ext_mean']) <= PRAUC_TOLERANCE_REC
        )
    else:
        return pd.Series({'decision': 'pendiente', 'reason': 'objective debe ser accuracy o recall'})

    if primary_ok and guardrail_ok and not unstable:
        decision = 'aceptar'
        reason = 'mejora primaria, respeta guardrails y mantiene estabilidad'
    else:
        decision = 'rechazar'
        reason_parts = []
        if not primary_ok:
            reason_parts.append('no mejora metrica primaria')
        if not guardrail_ok:
            reason_parts.append('viola guardrails secundarios')
        if unstable:
            reason_parts.append('incrementa inestabilidad entre pliegues')
        reason = '; '.join(reason_parts)

    return pd.Series({'decision': decision, 'reason': reason})


# Uso esperado:
# 1) Rellenar fase2_feature_eval_template.csv con resultados reales de CV por candidata.
# 2) Cargar ese archivo y aplicar la funcion para obtener la decision automatica.

# Ejemplo de pipeline de decision (listo para usar cuando haya metricas):
# eval_df = pd.read_csv(TABLES_PATH / 'fase2_feature_eval_template.csv')
# decisions_df = eval_df.join(eval_df.apply(decide_feature_candidate, axis=1))
# display(decisions_df)
# decisions_df.to_csv(TABLES_PATH / 'fase2_feature_eval_decisions.csv', index=False)

In [10]:
eval_df = pd.read_csv(TABLES_PATH / 'fase2_feature_eval_template.csv')

metric_cols = [
    'acc_base_mean', 'acc_ext_mean',
    'recall_base_mean', 'recall_ext_mean',
    'precision_base_mean', 'precision_ext_mean',
    'prauc_base_mean', 'prauc_ext_mean',
    'primary_std_base', 'primary_std_ext'
]

if eval_df[metric_cols].isna().all().all():
    print('No hay metricas cargadas aun en fase2_feature_eval_template.csv.')
    print('Paso siguiente: rellenar las metricas de CV (base vs extendido) y volver a ejecutar esta celda.')


decisions_df = eval_df.join(eval_df.apply(decide_feature_candidate, axis=1))
display(decisions_df)
decisions_df.to_csv(TABLES_PATH / 'fase2_feature_eval_decisions.csv', index=False)
print('Decisiones exportadas en:', (TABLES_PATH / 'fase2_feature_eval_decisions.csv').resolve())

No hay metricas cargadas aun en fase2_feature_eval_template.csv.
Paso siguiente: rellenar las metricas de CV (base vs extendido) y volver a ejecutar esta celda.


,candidate,objective,acc_base_mean,acc_ext_mean,recall_base_mean,recall_ext_mean,precision_base_mean,precision_ext_mean,prauc_base_mean,prauc_ext_mean,primary_std_base,primary_std_ext,decision,reason
0,total_minutes,accuracy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,pendiente,faltan metricas de validacion
1,total_minutes,recall,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,pendiente,faltan metricas de validacion
2,total_calls,accuracy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,pendiente,faltan metricas de validacion
3,total_calls,recall,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,pendiente,faltan metricas de validacion
4,minutes_per_call,accuracy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,pendiente,faltan metricas de validacion
5,minutes_per_call,recall,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,pendiente,faltan metricas de validacion
6,international_share_minutes,accuracy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,pendiente,faltan metricas de validacion
7,international_share_minutes,recall,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,pendiente,faltan metricas de validacion
8,log1p_v2,accuracy,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,pendiente,faltan metricas de validacion
9,log1p_v2,recall,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,pendiente,faltan metricas de validacion


Decisiones exportadas en: D:\Copias de seguridad\2025-2026\Master UCM\Módulo 11\Tarea\outputs\tables\fase2_feature_eval_decisions.csv
